In [1]:
# 1. Install split-folders
!pip install split-folders -q

import os
import shutil
import splitfolders
from google.colab import drive

# 2. Mount Google Drive
drive.mount('/content/drive')

# Set working paths (adjust this path to match your folder in Google Drive)
DRIVE_FOLDER = '/content/drive/MyDrive/Research Dataset'
LOCAL_EXTRACT = '/content/raw_data'
FINAL_DATASET = '/content/dataset'

os.makedirs(LOCAL_EXTRACT, exist_ok=True)
os.makedirs(FINAL_DATASET, exist_ok=True)

# 3. Unzip files to local Colab storage
print("Extracting datasets to Colab...")
!unzip -q "{DRIVE_FOLDER}/sapphires.zip" -d "{LOCAL_EXTRACT}/sapphires"
!unzip -q "{DRIVE_FOLDER}/other_gemstone.zip" -d "{LOCAL_EXTRACT}/other_raw"
!unzip -q "{DRIVE_FOLDER}/not_gemstone.zip" -d "{LOCAL_EXTRACT}/not_raw"

# 4. Prepare undivided images for split-folders
# Place images under subfolders named by class
TEMP_UNSPLIT = '/content/temp_to_split'
os.makedirs(f"{TEMP_UNSPLIT}/other_gemstone", exist_ok=True)
os.makedirs(f"{TEMP_UNSPLIT}/not_gemstone", exist_ok=True)

# Move Other_Gemstone and Not_Gemstone images into temp folders
for root, _, files in os.walk(f"{LOCAL_EXTRACT}/other_raw"):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            shutil.copy(os.path.join(root, f), f"{TEMP_UNSPLIT}/other_gemstone/{f}")

for root, _, files in os.walk(f"{LOCAL_EXTRACT}/not_raw"):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            shutil.copy(os.path.join(root, f), f"{TEMP_UNSPLIT}/not_gemstone/{f}")

# 5. Split remaining classes (70% train, 15% val, 15% test)
print("Splitting Other and Not Gemstone datasets...")
splitfolders.ratio(TEMP_UNSPLIT, output="/content/split_output", seed=42, ratio=(0.7, 0.15, 0.15))

# 6. Build the unified master directory
for split in ['train', 'val', 'test']:
    for cls in ['blue_sapphire', 'yellow_sapphire', 'other_gemstone', 'not_gemstone']:
        os.makedirs(f"{FINAL_DATASET}/{split}/{cls}", exist_ok=True)

# Copy the newly split other_gemstone & not_gemstone into the final directory
for split_name, target_name in [('train', 'train'), ('val', 'val'), ('test', 'test')]:
    for cls in ['other_gemstone', 'not_gemstone']:
        src = f"/content/split_output/{split_name}/{cls}"
        dst = f"{FINAL_DATASET}/{target_name}/{cls}"
        for f in os.listdir(src):
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))

# Copy already split sapphire images
# Maps any common directory name variation to the final directory
for root, _, files in os.walk(f"{LOCAL_EXTRACT}/sapphires"):
    root_lower = root.lower()
    for f in files:
        if not f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            continue

        # Determine split
        if 'train' in root_lower:
            target_split = 'train'
        elif 'val' in root_lower or 'validation' in root_lower:
            target_split = 'val'
        elif 'test' in root_lower:
            target_split = 'test'
        else:
            continue

        # Determine class
        if 'blue' in root_lower:
            target_cls = 'blue_sapphire'
        elif 'yellow' in root_lower:
            target_cls = 'yellow_sapphire'
        else:
            continue

        shutil.copy(os.path.join(root, f), f"{FINAL_DATASET}/{target_split}/{target_cls}/{f}")

# Print final sample counts per partition
print("\n--- Final Dataset Summary ---")
for split in ['train', 'val', 'test']:
    print(f"[{split.upper()}]")
    for cls in ['blue_sapphire', 'yellow_sapphire', 'other_gemstone', 'not_gemstone']:
        count = len(os.listdir(f"{FINAL_DATASET}/{split}/{cls}"))
        print(f"  {cls}: {count} images")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting datasets to Colab...
Splitting Other and Not Gemstone datasets...


Copying files: 2681 files [00:00, 3943.52 files/s]



--- Final Dataset Summary ---
[TRAIN]
  blue_sapphire: 686 images
  yellow_sapphire: 353 images
  other_gemstone: 840 images
  not_gemstone: 1036 images
[VAL]
  blue_sapphire: 147 images
  yellow_sapphire: 75 images
  other_gemstone: 180 images
  not_gemstone: 222 images
[TEST]
  blue_sapphire: 147 images
  yellow_sapphire: 77 images
  other_gemstone: 180 images
  not_gemstone: 223 images


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 1. Load datasets from directory structure
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{FINAL_DATASET}/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{FINAL_DATASET}/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{FINAL_DATASET}/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

class_names = train_ds.class_names
print("Class order:", class_names)

# Optimize pipeline speed
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# 2. Compute Class Weights for Imbalance (e.g., fewer yellow sapphires)
labels = []
for _, y in train_ds.unbatch():
    labels.append(np.argmax(y.numpy()))

computed_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(computed_weights))
print("Computed Class Weights:", class_weights)

# 3. Model Architecture with Pre-trained MobileNetV2
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.08),  # subtle rotations
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze backbone for initial training

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Phase 1 Training (Feature Extraction)
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_gem_model.keras', save_best_only=True, monitor='val_loss')
]

print("\nStarting Phase 1 Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weights,
    callbacks=callbacks
)

# 5. Phase 2: Fine-Tuning (Unfreeze top layers for better accuracy)
base_model.trainable = True
for layer in base_model.layers[:-30]:  # Only train the last 30 layers
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Lower learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nStarting Phase 2 Fine-Tuning...")
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weights,
    callbacks=callbacks
)

# 6. Final Evaluation on Unseen Test Dataset
print("\nEvaluating on Test Set:")
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

# ==========================================
# STEP 5 CODE ADDED BELOW
# ==========================================

# Save the model to your Google Drive folder
model.save(f"{DRIVE_FOLDER}/gemstone_classifier_model.keras")
print(f"Model saved to {DRIVE_FOLDER}")

# Function to predict on any new image
def predict_image(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = 100 * np.max(predictions[0])

    print(f"Prediction: {predicted_class} ({confidence:.2f}% confidence)")

Found 2915 files belonging to 4 classes.
Found 624 files belonging to 4 classes.
Found 627 files belonging to 4 classes.
Class order: ['blue_sapphire', 'not_gemstone', 'other_gemstone', 'yellow_sapphire']
Computed Class Weights: {0: np.float64(1.0623177842565599), 1: np.float64(0.703426640926641), 2: np.float64(0.8675595238095238), 3: np.float64(2.064447592067989)}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

Starting Phase 1 Training...
Epoch 1/12
92/92 ━━━━━━━━━━━━━━━━━━━━ 26s 143ms/step - accuracy: 0.7955 - loss: 0.5649 - val_accuracy: 0.9311 - val_loss: 0.2380
Epoch 2/12
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 111ms/step - accuracy: 0.9170 - loss: 0.2394 - val_accuracy: 0.9487 - val_loss: 0.1761
Epoch 3/12
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 128ms/step - accuracy: 0.9338 - loss: 0.1990 - val_accuracy: 0.9391 - val_loss: 0.1669
Epoch 4/12
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 124ms/step - accuracy: 0.9400 - loss: 0.1699 - val_accuracy: 0.9391 - val_loss: 0.1541
Epoch 5/12
92/92 ━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install gradio -q

import gradio as gr
import tensorflow as tf
import numpy as np

# A slightly modified version of your predict function to work with Gradio
def predict_gemstone(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = 100 * np.max(predictions[0])

    return f"Prediction: {predicted_class.upper()}\nConfidence: {confidence:.2f}%"

# Create the visual interface
demo = gr.Interface(
    fn=predict_gemstone,
    inputs=gr.Image(type="filepath"),
    outputs="text",
    title="Gemstone Classifier",
    description="Upload an image to see if it's a Blue Sapphire, Yellow Sapphire, Other Gemstone, or Not a Gemstone."
)

# Launch the interface inside the notebook
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d1af1b1b47232e4b14.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
